In [ ]:
import os  # to read the file size
import time  # to measure the prediction speed
# the final model is the tuned winner (the assert checks it is really a Pipeline)
final_pipeline = tuned_winner
assert isinstance(final_pipeline, Pipeline), "Final object must be a Pipeline"
# save preprocessing + model together in ONE file
joblib.dump(final_pipeline, "final_pipeline.joblib")
# load it back, to prove the saved file itself works
loaded_final = joblib.load("final_pipeline.joblib")
assert isinstance(loaded_final, Pipeline), "Loaded object must be a Pipeline"
# predict the test set with the loaded file, without refitting
final_pred = loaded_final.predict(X_test)
# the winner with default settings, to see what tuning added
winner_default = build_pipeline(winner_name).fit(X_train_final, y_train_final)
winner_default_pred = winner_default.predict(X_test)
# the final results table: baseline vs tuned (plus the steps in between)
results_final = pd.DataFrame([
    {"Model": "Baseline (Linear Regression)", **regression_metrics(y_test, baseline_pred)},
    {"Model": "Tuned Ridge", **regression_metrics(y_test, tuned_pred)},
    {"Model": f"{winner_name} (default settings)", **regression_metrics(y_test, winner_default_pred)},
    {"Model": f"{winner_name} (tuned, final)", **regression_metrics(y_test, final_pred)},
])
print("Held-out test results:")
print(results_final.round(4).to_string(index=False))
# a slow model is hard to deploy, so measure the prediction time
start = time.perf_counter()
loaded_final.predict(X_test.iloc[:10000])
elapsed = time.perf_counter() - start
print(f"\nPredicting 10,000 trips takes {elapsed:.2f} s")
# a huge file is hard to share, so check the size
print(f"File size: {os.path.getsize('final_pipeline.joblib') / 1e6:.1f} MB")
# the file must be loaded later with the same scikit-learn version
import sklearn
print("Pipeline steps:", [name for name, _ in loaded_final.steps])
print("Saved file: final_pipeline.joblib | scikit-learn", sklearn.__version__)